## Coding Challenge 5: Web scrapping of an API: COMTRADE

In this notebook, I connect to the Comtrade API to download information of Chilean exports, disagreggated by product.

Based on: https://github.com/uncomtrade/comtradeapicall/blob/main/tests/example%20calling%20functions%20-%20notebook.ipynb

In [1]:
pip install -q comtradeapicall # Installing COMTRADE package to download their API data.

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#'

[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install -q squarify

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly
import plotly.io as pio
import squarify
import requests
import comtradeapicall
import json

In [4]:
subscription_key = 'f2f03f3dc19047f8a1d1e960fccb3ed1' # comtrade api subscription key (from comtradedeveloper.un.org)
directory = '<OUTPUT DIR>'  # output directory for downloaded files 
proxy_url = '<PROXY URL>'  # optional if you need proxy url

# Where does Chile exports go?

Type of Product – Commodities (C) or Services (S)

Frequency – Annual (A) or Monthly (M) 

Classification Code – HS, SITC, BEC for Goods, EBOPS for Services

Period – YYYY for Annual or YYYYMM for Monthly

Reporter – All or Individual Reporters

Publication Date – Publication date range

aggregateBy - The level of disaggregation that you want in the products

### API call

In [124]:
#Here, I call the API for Chilean Exports

exp_chile = comtradeapicall.getFinalData(subscription_key=subscription_key, typeCode='C', freqCode='A', clCode='HS', period='2023',
                                        reporterCode='152', cmdCode='TOTAL', flowCode='X', partnerCode=None, #152 is Chile's country code
                                        partner2Code=None,
                                        customsCode=None, motCode=None, maxRecords=None, format_output='JSON',
                                        aggregateBy=None, breakdownMode='classic', countOnly=None, includeDesc=True)

### Continents

In [104]:
continent = pd.read_csv('../data/location_group_member.csv')

continent = continent[continent['group_type'] == 'continent'] # Just continents

# Rename key variable 
continent.rename(columns={
    'country_id': 'partnerCode'
},inplace=True)

In [ ]:
continent

### Cleaning and filtering process

In [125]:
# Drop World
exp_chile = exp_chile[exp_chile['partnerDesc'] !='World']


# Merge with continents df: 
exp_chile = pd.merge(exp_chile, continent, on="partnerCode", how='left')

# Rename key variable 
exp_chile.rename(columns={
    'group_name': 'continent',
    'partnerDesc': 'country'
},inplace=True)

# In share of total:
exp_chile['percent_fobvalue'] = exp_chile['fobvalue'] / exp_chile['fobvalue'].sum()
exp_chile = exp_chile.reset_index()

# Keeping only important variables
exp_chile = exp_chile[['continent', 'country','fobvalue', 'percent_fobvalue']]


# Fixing not merged countries
exp_chile.loc[exp_chile['country'] == 'USA', 'continent'] = 'North America'
exp_chile.loc[exp_chile['country'] == 'France', 'continent'] = 'Europe'
exp_chile.loc[exp_chile['country'] == 'Switzerland', 'continent'] = 'Europe'
exp_chile.loc[exp_chile['country'] == 'India', 'continent'] = 'Asia'
exp_chile.loc[exp_chile['country'] == 'Other Asia, nes', 'continent'] = 'Asia'
exp_chile.loc[exp_chile['country'] == 'Norway', 'continent'] = 'Europe'

exp_chile = exp_chile[exp_chile['continent'].isna() !=True]


In [105]:
exp_chile

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,NaN,False,0.0,False,None,9.493553e+10,9.493553e+10,0,False,True
1,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,0.0,True,0.0,False,None,2.189554e+04,2.189554e+04,4,False,True
2,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,0.0,True,0.0,False,None,4.317648e+05,4.317648e+05,4,False,True
3,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,0.0,True,0.0,False,None,7.860994e+06,7.860994e+06,4,False,True
4,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,0.0,True,0.0,False,None,1.120304e+05,1.120304e+05,4,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,0.0,True,0.0,False,None,8.095699e+07,8.095699e+07,4,False,True
178,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,0.0,True,0.0,False,None,3.820036e+05,3.820036e+05,4,False,True
179,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,NaN,False,0.0,False,None,1.301687e+05,1.301687e+05,0,False,True
180,C,A,20230101,2023,52,2023,152,CHL,Chile,X,...,0.0,True,0.0,False,None,1.760012e+06,1.760012e+06,4,False,True


### Plotting

In [127]:
# Define custom colors for each cluster_name
custom_colors = {
        'Asia': '#4E79A7',
        'North America': '#76B7B2',
        'South America': '#FF9DA7',
        'Europe': '#59A14F',
        'Oceania': '#F28E2B',
        'Africa': '#E15759'
        }

# Tableau colors
#4E79A7 (Blue)
#F28E2B (Orange)
#E15759 (Red)
# #76B7B2 (Teal)
# #59A14F (Green)
# #EDC949 (Yellow)
# #AF7AA1 (Purple)
#FF9DA7 (Pink)

#Plotting a tree map
fig = px.treemap(exp_chile, path=['country'],
                 values='percent_fobvalue',
                 color='continent',
                 color_discrete_map=custom_colors
                )

# Add title and subtitle
fig.update_layout(
    title={
        'text': " Where did Chilean Exports go in 2023?",
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 24} 
    },
    annotations=[
        dict(
            text="% of total FOB value exported",
            x=0.5,
            y=1.15,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=20)
        ),
          dict(
            text="Source: COMTRADE API",
            x=0,
            y=-0.4,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=100)
        )
    ],
    margin=dict(t=100, l=25, r=25, b=25),
    showlegend=True
)

# Erase the border of the treemap graph
fig.update_layout(xaxis_showgrid=False, yaxis_showgrid=False)

# Show percentages in each quadrant
fig.update_traces(textinfo='label+percent entry', textfont_size=20)

# Create white borders between categories
fig.update_traces(marker=dict(line=dict(width=1)))

# Remove all tooltips except the label
fig.update_traces(hovertemplate='Country: %{label}<br>Continent: %{customdata[0]}<extra></extra><br>Share of exports:%{value:.1%}')

# Saving
pio.write_json(fig, '../figures/figure5.json', pretty=True) # JSON
pio.write_html(fig, '../figures/figure5.html') #HTML 

fig.show()